# Chapter 2 — Burgers (sawtooth): weak-form losses — `weak_projection` vs `agnostic`, TD-PINNs vs Vanilla

**Case:** 1-D viscous Burgers, sawtooth analytic solution (Hopf–Cole) on $[0, 2\pi]$,
$\;u_t + u\,u_x = \nu\,u_{xx}$ — the same problem as the Chapter 1 TD-PINNs vs Vanilla
comparison, now varying the **loss form** instead of (or in addition to) the training scheme.

**Four combinations compared** (2 training schemes × 2 loss forms):

| | `weak_projection` | `agnostic` |
|---|---|---|
| **TD-PINNs** | Petrov–Galerkin projection of the Crank–Nicolson residual onto Legendre test functions | hand-derived reduced-order ($\nu>0$) weak Crank–Nicolson residual, spatial quadrature |
| **Vanilla**  | Petrov–Galerkin projection of the strong residual onto Legendre test functions | hand-derived reduced-order ($\nu>0$) weak residual, space–time Monte Carlo |

**Loss forms** (`PINLossFunction`, see `pinns_framework/loss.py`):

- **`weak_projection`** (renamed from `weak`) — the framework projects the residual onto a bank
  of $K$ Legendre test functions $\{v_k\}$ and minimises
  $$L_{\text{weak\_projection}} = \frac1K\sum_{k=1}^K\left(\underbrace{\textstyle\frac1N\sum_{i=1}^N R(x_i)\,v_k(x_i)}_{\text{MC estimate of }\int_\Omega R\,v_k\,dx}\right)^{\!2}$$
  `residual_fn` is **unchanged** from the strong form — it still returns the raw pointwise
  residual $R(x_i)$; only the aggregation from $R \to$ loss differs.

- **`agnostic`** (new) — `residual_fn` returns the *finished scalar loss* itself; `PINLossFunction`
  performs no reduction or projection at all. Both `agnostic` losses below implement the **full
  $\nu>0$ reduced-order weak form**: integrating the diffusion term $\nu u_{xx}$ by parts against
  a test function $v$ that **vanishes at the spatial boundary** removes the boundary flux term
  entirely, so only $u_x$ (not $u_{xx}$) is ever computed:
  $$\int_\Omega \nu\,u_{xx}\,v\,dx \;=\; \underbrace{\big[\nu\,u_x\,v\big]_{\partial\Omega}}_{=\,0\ (v=0\text{ at }\partial\Omega)} - \int_\Omega \nu\,u_x\,v_x\,dx \;=\; -\int_\Omega \nu\,u_x\,v_x\,dx$$
  The Dirichlet BC itself is still enforced the usual way, by the trainer's separate
  `bc_weight`-weighted MSE term — the vanishing test function only removes the *flux* boundary
  term produced by integration by parts, it is not what imposes $u=u_{bc}$.

  **TD-PINNs vs Vanilla use different quadrature**, dictated by how each scheme actually samples
  collocation points in `PINTrainer` (see Section 3 for the derivation and why): TD-PINNs gets a
  deterministic 1-D trapezoidal quadrature per step (points are all at the same time already);
  Vanilla needs a Monte Carlo space–time quadrature with a *product* test basis
  $v_k(x)\psi_m(t)$, because `PINTrainer._fit_vanilla` samples $t$ independently at random per
  collocation point — no two points reliably share a $t$, so a per-time-slice spatial quadrature
  is not available.

**What we report** (same diagnostics as the Chapter 1 notebook):
1. **All error norms** $L_1$, $L_2$, $L_\infty$ vs the analytical solution, per time level.
2. **Conservation diagnostics** — mass $M(t)=\int u\,dx$ (conserved) and energy
   $E(t)=\int u^2\,dx$ (dissipates: $\frac{dE}{dt}=-2\nu\int u_x^2\,dx\le 0$).
3. **Spectral test** — FFT in $x$ against the analytical solution, error per frequency band
   (DC / low / mid / high) — the sawtooth's shock lives in the high modes.


In [ ]:
import sys, os, time

ROOT = '/home/df.gomezb/Tesis'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from pinns_framework import (
    PINArchitecture, PINSpatialDimension, PINLossFunction, PINTrainer,
    compute_metrics, collocation_uniform,
)

torch.manual_seed(0)
np.random.seed(0)

# distinct serif / Computer-Modern look for every figure
plt.rcParams.update({
    'font.family'     : 'serif',
    'font.serif'      : ['DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size'       : 12,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 12,
    'legend.fontsize' : 9,
})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__, '| device:', DEVICE)

# marker / colour convention for the four (scheme, loss form) combinations
METHODS = {
    'TD-PINNs (weak_projection)': ('o', 'tab:blue'),
    'TD-PINNs (agnostic)':        ('^', 'tab:cyan'),
    'Vanilla (weak_projection)':  ('s', 'tab:orange'),
    'Vanilla (agnostic)':         ('D', 'tab:red'),
}

FIGDIR = 'figs'
os.makedirs(FIGDIR, exist_ok=True)


## 1. Problem setup — sawtooth analytic solution

In [ ]:
NU     = 0.07
X0, XF = 0.0, 2 * np.pi

def toothsaw(t, x):
    # Hopf-Cole analytic solution for viscous Burgers on [0, 2pi].
    x  = np.asarray(x, dtype=float)
    nu = NU
    c  = 4 * nu * (t + 1)
    a  = (-4*t + x)**2           / c
    b  = (-4*t + x - 2*np.pi)**2 / c
    num = ((-8*t + 2*x)            * np.exp(-a)
         + (-8*t + 2*x - 4*np.pi) * np.exp(-b)) / c
    den = np.exp(-b) + np.exp(-a)
    return 2 * nu * num / den + 4

x_plot = np.linspace(X0, XF, 500)
fig, ax = plt.subplots(figsize=(8, 3.5))
cmap = plt.cm.plasma
for i, t in enumerate([0.0, 0.01, 0.02, 0.025]):
    ax.plot(x_plot, toothsaw(t, x_plot), color=cmap(i / 3), lw=1.8, label=f't = {t:.3f}')
ax.set_xlabel('x'); ax.set_ylabel('u')
ax.set_title('Sawtooth analytic solution  (nu = 0.07)')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2. Experiment configuration

TD-PINNs uses a spatial-only network $u(x)$ per step (`layers[0]=1`); Vanilla uses a space–time
network $u(t,x)$ (`layers[0]=2`). Each scheme trains **two independently-initialised copies** of
its architecture — one per loss form (`weak_projection`, `agnostic`) — so the four runs share
capacity, collocation strategy, and optimiser schedule, and differ only in the loss form.


In [ ]:
PARAMS_TD  = {'layers': (1, 32, 32, 32, 1), 'activation': 'tanh'}   # u(x)  per step
PARAMS_VAN = {'layers': (2, 32, 32, 32, 1), 'activation': 'tanh'}   # u(t, x)

DT     = 0.005
T_END  = 0.025
NT     = round(T_END / DT)
times  = [k * DT for k in range(NT + 1)]

# optimisation (shared schedule across all four runs)
LR, MAX_LR, MIN_LR = 1e-2, 1e-2, 1e-6
_norm      = (15**2 + 5**2 + 1**2) ** 0.5
IC_WEIGHT  = 15.0 / _norm
BC_WEIGHT  = 5.0  / _norm
PDE_WEIGHT = 1.0  / _norm
N_COLLOC   = 500
N_EVAL     = 1000
N_TEST     = 10        # weak_projection: Legendre test functions per dim
                        # agnostic: vanishing Legendre test functions in x (both TD and Vanilla)
N_TEST_T   = 5          # agnostic (Vanilla only): plain Legendre test functions in t

TD_EPOCHS,  TD_TOL  = 100000, 1e-6     # CN per-step (converges)
VAN_EPOCHS, VAN_TOL = 30000,  1e-6     # space-time fit (uses full budget)

# vanilla_agnostic_loss uses the two-sample (U-statistic) trick (Section 3), which is an
# unbiased but NOT nonnegative estimator of the true squared residual -- it can dip slightly
# below 0 on noisy batches even far from convergence. PINTrainer._run_epochs stops as soon as
# loss.item() < tolerance, so a normal positive VAN_TOL would trigger a spurious "converged"
# the first time a batch happens to land negative, badly undertraining this one run. Give it an
# unreachable tolerance instead and let it use the full epoch budget like the others.
VAN_TOL_AGN = -1e9

print(f'{NT} steps, dt={DT}, T={T_END}')


## 3. IC / BC, residuals, exact evaluator, time-slice wrapper

### Strong residuals (`weak_projection` reuses these unchanged)

- **`td_residual`** (Crank–Nicolson): $\frac{u-u^{t-1}}{\Delta t} + \tfrac12[S(u)+S(u^{t-1})]$,
  $S(w)=w w_x-\nu w_{xx}$.
- **`vanilla_residual`**: $u_t + u u_x - \nu u_{xx}$, with $u_t$ from autograd in $t$.

### Agnostic weak-form losses (`form='agnostic'`) — full $\nu>0$ physics

Both derivations test the strong residual against a **vanishing Legendre test function**
$v_k(\xi)=P_{k+1}(\xi)-P_{k-1}(\xi)$, $\xi=\frac{2(x-X_0)}{X_F-X_0}-1\in[-1,1]$, which satisfies
$v_k(\pm1)=0$ (hp-VPINN's standard choice). Two useful identities fall out for free from the
Legendre three-term recurrence:
$$v_k(\pm1)=0 \qquad\text{and}\qquad \frac{dv_k}{d\xi}=(2k+1)P_k(\xi)$$
The first kills the boundary-flux term from integrating $\nu u_{xx}$ by parts (Section 0); the
second means $v_k'$ costs nothing extra to evaluate — no autograd through the test functions.
Convective flux $F(u)=u^2/2$ is treated the same way ($u u_x = F(u)_x$), so $u u_x$ is also
integrated by parts and never appears directly.

**TD-PINNs (`td_agnostic_loss`)** — for each step, `PINTrainer._sample_colloc` returns
collocation points that already all belong to the *same* time step (TD-PINNs has no time input;
time is handled by the outer stepping loop). So the spatial integral is well posed with an
ordinary 1-D quadrature: sort the collocation points and apply the trapezoidal rule. For each
test function $v_k$:
$$R_k=\int\frac{u-u^{n}}{\Delta t}v_k\,dx+\tfrac12\Big[\!-\!\!\int F(u)v_k'dx+\nu\!\!\int u_xv_k'dx\Big]+\tfrac12\Big[\!-\!\!\int F(u^{n})v_k'dx+\nu\!\!\int u^{n}_xv_k'dx\Big]$$
with the $u^n$ (previous-step) branch detached, matching the trainer's existing Backward/CN
convention (`u_prev.detach()`). $L=\text{mean}_k R_k^2$.

**Vanilla (`vanilla_agnostic_loss`)** — `PINTrainer._fit_vanilla` samples
`t_col = torch.rand(n_col, 1) * T`: **every collocation point gets its own independent random
$t$**. A spatial quadrature needs several $x$ points sharing one $t$, which this sampler never
produces (each "slice" would have exactly 1 point). The fix is to test against a **product basis**
$v_k(x)\,\psi_m(t)$ ($\psi_m$ = plain, non-vanishing Legendre polynomials in $t$ — legitimate
*only* because $u_t$ is kept strong/autodiff rather than integrated by parts in time, so no
temporal boundary term ever appears; $v_k$ in $x$ still keeps the vanishing constraint, since that
one *is* required) and Monte-Carlo-integrate the whole space–time double integral directly —
valid because it's now a single integral over the full $(t,x)$ rectangle, the same MC scheme
`weak_projection` already uses, just in 2-D. $t$ is rescaled to $\tau=2t/T_{\text{end}}-1\in[-1,1]$
before evaluating $\psi_m$, exactly mirroring what `_spatial_weak_setup` already does for $x$
(Legendre blows up outside $[-1,1]$):
$$R_{km}\approx\frac1N\sum_{i=1}^N\Big(u_{t,i}\,v_k(x_i)\psi_m(t_i)-F(u_i)\,v_k'(x_i)\psi_m(t_i)+\nu\,u_{x,i}\,v_k'(x_i)\psi_m(t_i)\Big)$$
(As with `weak_projection`, the missing $|\Omega|\!\cdot\!T$ volume constant is a fixed scalar
absorbed into `pde_weight` — see the earlier discussion on Monte Carlo integration in this
framework.)

Two refinements on top of the basic scheme above, both applied in the code:

- **Test-function normalisation.** $v_k$ and $\psi_m$ are both scaled to unit $L^2([-1,1])$ norm
  (closed form: $\|v_k\|^2=\tfrac{2}{2k+3}+\tfrac{2}{2k-1}$ since $P_{k+1}\perp P_{k-1}$;
  $\|\psi_m\|^2=\tfrac{2}{2m+1}$). Raw Legendre norms decay with degree, so with a *product* basis
  ($K_x\times K_t$ residuals) the un-normalised low-order modes would dominate the loss by an
  order of magnitude and the optimiser would under-weight the higher modes that actually see the
  shock. `_vanishing_tests` normalises this way for **both** TD and Vanilla, so the two schemes
  use a consistent test-function convention.
- **Two-sample (U-statistic) squaring.** A single-batch MC estimate $\hat R_{km}$ is unbiased for
  $R_{km}$, but $\mathbb E[\hat R_{km}^2]=R_{km}^2+\operatorname{Var}(\hat R_{km})$ — squaring a
  noisy estimate adds a persistent, non-vanishing bias, one that can *dominate* right when
  $R_{km}^2$ itself gets small (late training, exactly the regime this notebook's error
  comparisons care about). The fix: split the batch into two independent halves $A,B$ and use
  $\hat R_{km}^{(A)}\hat R_{km}^{(B)}$ in place of $\hat R_{km}^2$ — unbiased for $R_{km}^2$
  since the halves are independent, same cost. `td_agnostic_loss` keeps plain squaring: its
  quadrature is deterministic trapezoidal over sorted points, not an i.i.d. Monte Carlo mean, so
  the same variance argument doesn't transfer with the same severity there.

This asymmetry — TD-PINNs gets exact per-step 1-D quadrature "for free," Vanilla is forced into a
noisier 2-D Monte Carlo scheme (with the two correction terms above) — is itself worth reporting:
it's a structural consequence of time-marching vs. global space–time training, not a modelling
choice.

- **`TimeSlice`** wraps the Vanilla $u(t,x)$ net into a spatial-only $u(x)$ at a fixed $t$, so the
  same `compute_metrics` path works for both schemes.


In [ ]:
x_ic_np = np.linspace(X0, XF, 201)
u_ic_np = toothsaw(0, x_ic_np)
x_ic = torch.tensor(x_ic_np, dtype=torch.float32).unsqueeze(1)
u_ic = torch.tensor(u_ic_np, dtype=torch.float32).unsqueeze(1)

u_bc_val = float(toothsaw(0, X0))
x_bc = torch.tensor([[X0], [XF]], dtype=torch.float32)
u_bc = torch.full((2, 1), u_bc_val, dtype=torch.float32)

x_dim = PINSpatialDimension('x', domain=(X0, XF), n_colloc=N_COLLOC)

# ── strong pointwise residuals — reused as-is for form='weak_projection' ────

def td_residual(u, u_prev, x, dt, nu):
    """TD-PINNs Crank-Nicolson residual."""
    def S(w):
        w_x  = torch.autograd.grad(w.sum(),   x, create_graph=True)[0]
        w_xx = torch.autograd.grad(w_x.sum(), x, create_graph=True)[0]
        return w * w_x - nu * w_xx
    return (u - u_prev.detach()) / dt + 0.5 * (S(u) + S(u_prev).detach())

def vanilla_residual(u, t, x, nu):
    """Vanilla strong-form Burgers residual u_t + u u_x - nu u_xx."""
    u_t  = torch.autograd.grad(u.sum(),   t, create_graph=True)[0]
    u_x  = torch.autograd.grad(u.sum(),   x, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
    return u_t + u * u_x - nu * u_xx

# ── agnostic (form='agnostic') weak-form machinery — full nu>0 physics ──────
#
# Shared spatial building blocks (both TD and Vanilla): vanishing Legendre
# test functions v_k(xi) = P_{k+1}(xi) - P_{k-1}(xi), which satisfy
# v_k(+-1)=0, so integrating nu*u_xx (and u*u_x = F(u)_x) by parts against
# v_k drops the boundary-flux term entirely -- only first derivatives of u
# are ever computed. dv_k/dxi = (2k+1) P_k comes for free from the Legendre
# recurrence (Bonnet's derivative identity), no autograd needed for it.

def _legendre_bank(xi, n_max):
    """P_0 .. P_{n_max+1} via the three-term recurrence. xi: (N,1) in [-1,1].
    Returns a list of (N,1) tensors, length n_max+2."""
    Ps = [torch.ones_like(xi), xi]
    for n in range(1, n_max + 1):
        Ps.append(((2 * n + 1) * xi * Ps[n] - n * Ps[n - 1]) / (n + 1))
    return Ps

def _vanishing_tests(xi, n_test):
    """v_k = P_{k+1} - P_{k-1}  (vanishes at xi=+-1), dv_k/dxi = (2k+1) P_k,
    for k=1..n_test, L2([-1,1])-normalised (||v_k||=1). Analytic norm:
    P_{k+1} perp P_{k-1}, so ||v_k||^2 = ||P_{k+1}||^2+||P_{k-1}||^2
    = 2/(2k+3) + 2/(2k-1) (||P_n||^2 = 2/(2n+1)). Normalising matters once
    several test functions are combined into one loss -- raw Legendre norms
    decay with degree, so without this the low-order modes would dominate
    and the optimiser would under-weight the higher modes that see the
    shock. xi: (N,1) -> V, dV_dxi: (N, n_test), both already normalised."""
    Ps = _legendre_bank(xi, n_test)
    V  = torch.cat([Ps[k + 1] - Ps[k - 1] for k in range(1, n_test + 1)], dim=1)
    dV = torch.cat([(2 * k + 1) * Ps[k]   for k in range(1, n_test + 1)], dim=1)
    k = torch.arange(1, n_test + 1, dtype=xi.dtype, device=xi.device)
    norm = torch.sqrt(2.0 / (2 * k + 3) + 2.0 / (2 * k - 1))
    return V / norm, dV / norm

def _legendre_plain(tau, n_max):
    """Plain (non-vanishing) P_0 .. P_{n_max-1}, L2([-1,1])-normalised
    (||P_m||=1; analytic norm ||P_m||^2 = 2/(2m+1)) for the same reason as
    _vanishing_tests. tau: (N,1) in [-1,1] -> (N, n_max)."""
    Ps = [torch.ones_like(tau)]
    if n_max > 1:
        Ps.append(tau.clone())
    for k in range(1, n_max - 1):
        Ps.append(((2 * k + 1) * tau * Ps[-1] - k * Ps[-2]) / (k + 1))
    Psi = torch.cat(Ps[:n_max], dim=1)
    m = torch.arange(0, n_max, dtype=tau.dtype, device=tau.device)
    norm = torch.sqrt(2.0 / (2 * m + 1))
    return Psi / norm

def _trapz_weights(x_sorted):
    """Trapezoid quadrature weights for sorted 1-D nodes. (N,1) -> (N,1)."""
    dx = x_sorted[1:] - x_sorted[:-1]
    w = torch.zeros_like(x_sorted)
    w[:-1] += dx / 2
    w[1:]  += dx / 2
    return w

def _spatial_weak_setup(x, n_test=N_TEST):
    """Sort 1-D collocation points; build vanishing Legendre test functions
    and trapezoid quadrature weights on the physical domain [X0, XF].
    Returns (sort_idx, V, dV_dx, w) -- V, dV_dx, w are detached (fixed
    w.r.t. network parameters, same convention as weak_projection's test
    matrix)."""
    idx = torch.argsort(x.view(-1))
    xs  = x.view(-1, 1)[idx]
    L   = XF - X0
    xi  = (2.0 * (xs - X0) / L - 1.0).detach()
    V, dV_dxi = _vanishing_tests(xi, n_test)
    dV_dx = dV_dxi * (2.0 / L)             # chain rule: d/dx = (2/L) d/dxi
    w = _trapz_weights(xs)
    return idx, V.detach(), dV_dx.detach(), w.detach()

def _project(w, f, G):
    """Quadrature estimate of integral(f * g_k dx) for every test function k.
    w, f: (N,1); G: (N,K) -> (K,)."""
    return (w * f * G).sum(dim=0)


def td_agnostic_loss(u, u_prev, x, dt, nu):
    """
    Weak-form Crank-Nicolson loss for TD-PINNs (form='agnostic'), nu > 0.

    For each vanishing Legendre test function v_k (k=1..N_TEST):
        R_k = INT (u - u_prev)/dt * v_k dx
              + 1/2 [ -INT F(u)      v_k' dx + nu INT u_x      v_k' dx ]
              + 1/2 [ -INT F(u_prev) v_k' dx + nu INT u_prev_x v_k' dx ]
    with F(w) = w^2/2 (so F(w)' = w w_x, the Burgers convective flux). The
    u_prev branch is detached, matching the trainer's existing convention
    (u_prev.detach() in td_residual above). All collocation points for a
    TD-PINNs step already share one time level (time is handled by the
    outer stepping loop, not the network), so an ordinary sorted-trapezoid
    quadrature in x is exact enough. loss = mean_k R_k^2.
    """
    u_x  = torch.autograd.grad(u.sum(),      x, create_graph=True)[0]
    up_x = torch.autograd.grad(u_prev.sum(), x, create_graph=True)[0]

    idx, V, dVdx, w = _spatial_weak_setup(x)
    us,  uxs  = u[idx],               u_x[idx]
    ups, upxs = u_prev.detach()[idx], up_x.detach()[idx]

    R  = _project(w, (us - ups) / dt, V)
    R += 0.5 * (-_project(w, 0.5 * us**2,  dVdx) + nu * _project(w, uxs,  dVdx))
    R += 0.5 * (-_project(w, 0.5 * ups**2, dVdx) + nu * _project(w, upxs, dVdx))
    return (R ** 2).mean()


def vanilla_agnostic_loss(u, t, x, nu):
    """
    Space-time weak-form loss for Vanilla PINNs (form='agnostic'), nu > 0.

    R_km = INT_0^T INT_0^L ( u_t v_k - F(u) v_k' + nu u_x v_k' ) psi_m dx dt
         ~= mean_i ( u_t,i v_k(x_i) - F(u_i) v_k'(x_i)
                      + nu u_x,i v_k'(x_i) ) psi_m(t_i)

    with F(u) = u^2/2. Weak-in-space only: u_t stays strong (autodiff, no
    boundary to integrate by parts in time -- which is exactly why psi_m
    (plain Legendre in t) needs no vanishing constraint, unlike v_k in x).
    nu*u_xx is integrated by parts against v_k, which DOES vanish at the
    spatial boundary, so u_xx is never computed.

    Why space-time Monte Carlo instead of a per-time-slice quadrature:
    PINTrainer._fit_vanilla samples t_col = torch.rand(n_col, 1) * T --
    every collocation point gets its own independent random t, so no group
    of points shares a single t to integrate over in x. Testing against the
    product basis v_k(x) psi_m(t) turns the residual into one genuine
    integral over the whole (t, x) rectangle, which plain Monte Carlo (the
    same scheme PINLossFunction uses for form='weak_projection') estimates
    correctly regardless of how t is distributed.

    Two-sample (U-statistic) squaring: a naive single-batch MC estimate
    R_hat = mean_i(f_km,i) satisfies E[R_hat] = R_km (unbiased) but
    E[R_hat^2] = R_km^2 + Var(R_hat) -- squaring a noisy estimate biases the
    loss upward by the estimator's own variance, which does NOT vanish at
    the true solution and can dominate once R_km^2 gets small (i.e. exactly
    the late-training regime this notebook's error comparisons care about).
    Splitting the batch into two independent halves A, B and using
    R_A * R_B instead of R_hat^2 fixes this: E[R_A * R_B] = R_km^2 exactly,
    since A and B are independent and each unbiased for R_km. Same cost,
    slightly noisier per-step gradient (half the points feed each factor).
    The per-batch value can go slightly negative -- expected for an
    unbiased-but-not-nonnegative estimator, harmless for SGD; don't use
    this loss value itself as a converged-error diagnostic (compute_metrics
    against the analytical solution, used throughout this notebook, is
    unaffected and remains the right diagnostic).
    """
    u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
    u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]

    L   = XF - X0
    xi  = (2.0 * (x - X0) / L - 1.0).detach()
    tau = (2.0 * t / T_END - 1.0).detach()

    V, dV_dxi = _vanishing_tests(xi, N_TEST)      # (N, Kx), unit-norm
    dV_dx = dV_dxi * (2.0 / L)
    Psi = _legendre_plain(tau, N_TEST_T)          # (N, Kt), unit-norm

    Phi     = (V.unsqueeze(2)     * Psi.unsqueeze(1)).reshape(x.shape[0], -1).detach()
    dPhi_dx = (dV_dx.unsqueeze(2) * Psi.unsqueeze(1)).reshape(x.shape[0], -1).detach()

    F_u = 0.5 * u ** 2
    f = u_t * Phi - F_u * dPhi_dx + nu * u_x * dPhi_dx   # (N, Kx*Kt) per-point integrand

    n = x.shape[0]
    perm = torch.randperm(n, device=x.device)
    half = n // 2
    idxA, idxB = perm[:half], perm[half:2 * half]
    R_A = f[idxA].mean(dim=0)
    R_B = f[idxB].mean(dim=0)
    return (R_A * R_B).mean()

loss_td_wp   = PINLossFunction([x_dim], td_residual,          form='weak_projection', n_test=N_TEST)
loss_van_wp  = PINLossFunction([x_dim], vanilla_residual,     form='weak_projection', n_test=N_TEST)
loss_td_agn  = PINLossFunction([x_dim], td_agnostic_loss,     form='agnostic')
loss_van_agn = PINLossFunction([x_dim], vanilla_agnostic_loss, form='agnostic')

def exact_fn(t, x_pts):
    u = toothsaw(t, x_pts.detach().cpu().numpy().ravel())
    return torch.tensor(u, dtype=torch.float32, device=x_pts.device).unsqueeze(1)

class TimeSlice(nn.Module):
    """Expose a space-time net u(t,x) as a spatial-only net u(x) at fixed t."""
    def __init__(self, net, t):
        super().__init__(); self.net = net; self.t = float(t)
    def forward(self, x):
        tcol = torch.full((x.shape[0], 1), self.t, device=x.device, dtype=x.dtype)
        return self.net(torch.cat([tcol, x], dim=1))


## 4. Train TD-PINNs — `weak_projection`

In [ ]:
torch.manual_seed(0); np.random.seed(0)
net_td_wp = PINArchitecture('MLP', PARAMS_TD)
trainer_td_wp = PINTrainer(
    net=net_td_wp, scheme='TD-PINNS', loss_fn=loss_td_wp, spatial_dims=[x_dim],
    epochs=TD_EPOCHS, dt=DT, lr=LR, max_lr=MAX_LR, min_lr=MIN_LR, tolerance=TD_TOL,
    ic_weight=IC_WEIGHT, bc_weight=BC_WEIGHT, pde_weight=PDE_WEIGHT,
    collocation_fn=collocation_uniform, device=DEVICE, verbose=True, log_every=20000,
)
t0 = time.perf_counter()
nets_td_wp = trainer_td_wp.fit(x_ic=x_ic, u_ic=u_ic, x_bc=x_bc, u_bc=u_bc, T=T_END, nu=NU)
wall_td_wp = time.perf_counter() - t0
epochs_td_wp = sum(h['epochs'] for h in trainer_td_wp.step_history[1:])
print(f'TD-PINNs (weak_projection) done | epochs(PDE steps)={epochs_td_wp} | wall {wall_td_wp:.1f}s')


## 5. Train TD-PINNs — `agnostic`

In [ ]:
torch.manual_seed(0); np.random.seed(0)
net_td_agn = PINArchitecture('MLP', PARAMS_TD)
trainer_td_agn = PINTrainer(
    net=net_td_agn, scheme='TD-PINNS', loss_fn=loss_td_agn, spatial_dims=[x_dim],
    epochs=TD_EPOCHS, dt=DT, lr=LR, max_lr=MAX_LR, min_lr=MIN_LR, tolerance=TD_TOL,
    ic_weight=IC_WEIGHT, bc_weight=BC_WEIGHT, pde_weight=PDE_WEIGHT,
    collocation_fn=collocation_uniform, device=DEVICE, verbose=True, log_every=20000,
)
t0 = time.perf_counter()
nets_td_agn = trainer_td_agn.fit(x_ic=x_ic, u_ic=u_ic, x_bc=x_bc, u_bc=u_bc, T=T_END, nu=NU)
wall_td_agn = time.perf_counter() - t0
epochs_td_agn = sum(h['epochs'] for h in trainer_td_agn.step_history[1:])
print(f'TD-PINNs (agnostic) done | epochs(PDE steps)={epochs_td_agn} | wall {wall_td_agn:.1f}s')


## 6. Train Vanilla PINNs — `weak_projection`

In [ ]:
torch.manual_seed(0); np.random.seed(0)
net_van_wp = PINArchitecture('MLP', PARAMS_VAN)
trainer_van_wp = PINTrainer(
    net=net_van_wp, scheme='VANILLA', loss_fn=loss_van_wp, spatial_dims=[x_dim],
    epochs=VAN_EPOCHS, lr=LR, max_lr=MAX_LR, min_lr=MIN_LR, tolerance=VAN_TOL,
    ic_weight=IC_WEIGHT, bc_weight=BC_WEIGHT, pde_weight=PDE_WEIGHT,
    collocation_fn=collocation_uniform, device=DEVICE, verbose=True, log_every=5000,
)
t0 = time.perf_counter()
vanilla_net_wp = trainer_van_wp.fit(x_ic=x_ic, u_ic=u_ic, x_bc=x_bc, u_bc=u_bc, T=T_END, nu=NU)[0]
wall_van_wp = time.perf_counter() - t0
epochs_van_wp = trainer_van_wp.step_history[-1]['epochs']
print(f'Vanilla (weak_projection) done | epochs={epochs_van_wp} | wall {wall_van_wp:.1f}s')


## 7. Train Vanilla PINNs — `agnostic`

In [ ]:
torch.manual_seed(0); np.random.seed(0)
net_van_agn = PINArchitecture('MLP', PARAMS_VAN)
trainer_van_agn = PINTrainer(
    # tolerance=VAN_TOL_AGN (not VAN_TOL): the two-sample estimator in vanilla_agnostic_loss
    # can dip slightly negative on noisy batches even far from convergence, which would
    # otherwise trigger a spurious early stop against a normal positive tolerance -- see the
    # config cell (Section 2) for the full explanation.
    net=net_van_agn, scheme='VANILLA', loss_fn=loss_van_agn, spatial_dims=[x_dim],
    epochs=VAN_EPOCHS, lr=LR, max_lr=MAX_LR, min_lr=MIN_LR, tolerance=VAN_TOL_AGN,
    ic_weight=IC_WEIGHT, bc_weight=BC_WEIGHT, pde_weight=PDE_WEIGHT,
    collocation_fn=collocation_uniform, device=DEVICE, verbose=True, log_every=5000,
)
t0 = time.perf_counter()
vanilla_net_agn = trainer_van_agn.fit(x_ic=x_ic, u_ic=u_ic, x_bc=x_bc, u_bc=u_bc, T=T_END, nu=NU)[0]
wall_van_agn = time.perf_counter() - t0
epochs_van_agn = trainer_van_agn.step_history[-1]['epochs']
print(f'Vanilla (agnostic) done | epochs={epochs_van_agn} | wall {wall_van_agn:.1f}s')


## 8. Evaluate all four runs — error norms ($L_1$, $L_2$, $L_\infty$)

All four are reduced to a list of per-time spatial nets (Vanilla via `TimeSlice`), evaluated with
the framework's `compute_metrics`; $L_1$ (mean-abs) is added per level.


In [ ]:
slices_van_wp  = [TimeSlice(vanilla_net_wp,  t).to(DEVICE) for t in times]
slices_van_agn = [TimeSlice(vanilla_net_agn, t).to(DEVICE) for t in times]

def add_L1(df, slice_nets):
    xg = torch.linspace(X0, XF, N_EVAL).unsqueeze(1).to(DEVICE)
    l1 = []
    for t, net in zip(times, slice_nets):
        net.eval()
        with torch.no_grad():
            up = net(xg)
            if up.dim() == 1:
                up = up.unsqueeze(1)
        l1.append(float((up - exact_fn(t, xg)).abs().mean()))
    df['L1'] = l1
    return df

results = {
    'TD-PINNs (weak_projection)': {'slices': nets_td_wp,    'epochs': epochs_td_wp,  'wall': wall_td_wp},
    'TD-PINNs (agnostic)':        {'slices': nets_td_agn,   'epochs': epochs_td_agn, 'wall': wall_td_agn},
    'Vanilla (weak_projection)':  {'slices': slices_van_wp, 'epochs': epochs_van_wp, 'wall': wall_van_wp},
    'Vanilla (agnostic)':         {'slices': slices_van_agn,'epochs': epochs_van_agn,'wall': wall_van_agn},
}
for m, r in results.items():
    df = compute_metrics(r['slices'], times, [x_dim], n_eval=N_EVAL,
                         exact_fn=exact_fn, device=DEVICE)
    r['df'] = add_L1(df, r['slices'])

rows = []
for m, r in results.items():
    d = r['df']
    rows.append({'method': m, 'epochs': r['epochs'],
                 'mean_L1': d['L1'].mean(), 'mean_L2': d['L2'].mean(), 'mean_Linf': d['Linf'].mean(),
                 'final_L1': d['L1'].iloc[-1], 'final_L2': d['L2'].iloc[-1], 'final_Linf': d['Linf'].iloc[-1],
                 'wall_s': r['wall']})
summary = pd.DataFrame(rows).set_index('method')
pd.set_option('display.float_format', '{:.4e}'.format)
print('=== weak_projection vs agnostic, TD-PINNs vs Vanilla — error summary (vs analytical) ===')
print(summary.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (label, col) in zip(axes, [(r'$L_1$', 'L1'), (r'$L_2$', 'L2'), (r'$L_\infty$', 'Linf')]):
    for m, r in results.items():
        mk, c = METHODS[m]
        ax.semilogy(times, r['df'][col].values, '-', marker=mk, ms=6, lw=1.5, color=c, label=m)
    ax.set_xlabel('t'); ax.set_ylabel(f'{label} error vs exact')
    ax.set_title(f'{label} error per time level')
    ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=7)
fig.suptitle('Burgers sawtooth — weak_projection vs agnostic, TD-PINNs vs Vanilla', fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'burgers_weakforms_errors.pdf'), bbox_inches='tight')
print('saved ->', os.path.join(FIGDIR, 'burgers_weakforms_errors.pdf'))
plt.show()


## 9. Conservation — mass $\int u\,dx$ and energy $\int u^2\,dx$

Mass should stay flat (it is conserved analytically: $\int u\,dx = 4\cdot 2\pi$). Energy must
**decrease** monotonically (viscous dissipation $dE/dt = -2\nu\int u_x^2\,dx \le 0$). The
analytical curve is the reference; a good method tracks it. Since the `agnostic` losses now
include the diffusion term (via the vanishing-test-function integration by parts, Section 3),
their energy curves are expected to dissipate correctly too — a divergence here would indicate a
genuine training or quadrature issue, not a missing physics term.


In [ ]:
df_ref = results['TD-PINNs (weak_projection)']['df']   # exact_mass / exact_energy are method-independent

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# mass conservation
ax = axes[0]
ax.plot(times, df_ref['exact_mass'].values, 'k-', lw=2.4, label='analytical')
for m, r in results.items():
    mk, c = METHODS[m]
    ax.plot(times, r['df']['mass'].values, '-', marker=mk, ms=6, lw=1.5, color=c, label=m)
ax.set_xlabel('t'); ax.set_ylabel(r'mass  $\int u\,dx$')
ax.set_title('Mass conservation (flat = conserved)')
ax.grid(alpha=0.3); ax.legend(fontsize=7)

# energy dissipation
ax = axes[1]
ax.plot(times, df_ref['exact_energy'].values, 'k-', lw=2.4, label='analytical')
for m, r in results.items():
    mk, c = METHODS[m]
    ax.plot(times, r['df']['energy'].values, '-', marker=mk, ms=6, lw=1.5, color=c, label=m)
ax.set_xlabel('t'); ax.set_ylabel(r'energy  $\int u^2\,dx$')
ax.set_title(r'Energy dissipation ($dE/dt \leq 0$, decreasing)')
ax.grid(alpha=0.3); ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'burgers_weakforms_conservation.pdf'), bbox_inches='tight')
print('saved ->', os.path.join(FIGDIR, 'burgers_weakforms_conservation.pdf'))
plt.show()

# numeric check
print('\nanalytical mass (should be constant):', np.round(df_ref['exact_mass'].values, 5))
print('analytical energy (should decrease) :', np.round(df_ref['exact_energy'].values, 4))


## 10. Spectral test — error per frequency band

On a uniform periodic grid over $[0,2\pi)$ we take the real FFT in $x$ of each run and of the
analytical solution at the final time, then look at the per-mode error
$|\hat u^{\text{pred}}_k - \hat u^{\text{exact}}_k|$ grouped into bands. The sawtooth's **shock**
lives in the **high** wavenumbers, so that band is the hard one — and the one most sensitive to
how well each loss form's quadrature (deterministic trapezoid for TD-PINNs, Monte Carlo for
Vanilla, Section 3) resolves the diffusion term there.


In [ ]:
N_SPEC = 512
x_spec = np.linspace(X0, XF, N_SPEC, endpoint=False)          # periodic grid
xs     = torch.tensor(x_spec, dtype=torch.float32).unsqueeze(1).to(DEVICE)
kf     = NT                                                   # final time index
t_spec = times[kf]

u_ex = toothsaw(t_spec, x_spec)
U_ex = np.fft.rfft(u_ex) / N_SPEC
kk   = np.arange(len(U_ex))                                   # wavenumbers 0..N/2

spec = {}
for m, r in results.items():
    net = r['slices'][kf].eval()
    with torch.no_grad():
        u = net(xs).cpu().numpy().ravel()
    spec[m] = np.fft.rfft(u) / N_SPEC

# frequency bands (inclusive wavenumber ranges)
BANDS = [(0, 0, 'DC (k=0)'), (1, 4, 'low (1-4)'), (5, 15, 'mid (5-15)'), (16, kk[-1], 'high (16+)')]
band_labels = [b[2] for b in BANDS]
band_err = {m: [np.abs(spec[m] - U_ex)[lo:hi + 1].sum() for lo, hi, _ in BANDS] for m in results}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) magnitude spectrum at final time
ax = axes[0]
ax.semilogy(kk[1:], np.abs(U_ex)[1:], 'k-', lw=2.0, label='analytical')
for m in results:
    mk, c = METHODS[m]
    ax.semilogy(kk[1:], np.abs(spec[m])[1:], '-', color=c, lw=1.3, label=m)
ax.set_xlabel('wavenumber k'); ax.set_ylabel(r'$|\hat u_k|$')
ax.set_title(f'Spectrum at t = {t_spec:.3f}'); ax.set_xlim(0, 40)
ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=7)

# (b) per-mode error
ax = axes[1]
for m in results:
    mk, c = METHODS[m]
    ax.semilogy(kk[1:], np.abs(spec[m] - U_ex)[1:], '-', color=c, lw=1.3, label=m)
ax.set_xlabel('wavenumber k'); ax.set_ylabel(r'$|\hat u^{pred}_k - \hat u^{exact}_k|$')
ax.set_title('Per-mode error'); ax.set_xlim(0, 40)
ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=7)

# (c) band-summed error bars
ax = axes[2]
xb = np.arange(len(BANDS)); w = 0.8 / len(results)
for i, m in enumerate(results):
    mk, c = METHODS[m]
    ax.bar(xb + (i - (len(results) - 1) / 2) * w, band_err[m], width=w, color=c, label=m)
ax.set_yscale('log')
ax.set_xticks(xb); ax.set_xticklabels(band_labels, rotation=15)
ax.set_ylabel('summed spectral error'); ax.set_title('Error per frequency band')
ax.grid(True, which='both', axis='y', alpha=0.3); ax.legend(fontsize=7)

fig.suptitle(f'Spectral error analysis (t = {t_spec:.3f})', fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'burgers_weakforms_spectral.pdf'), bbox_inches='tight')
print('saved ->', os.path.join(FIGDIR, 'burgers_weakforms_spectral.pdf'))
plt.show()

band_df = pd.DataFrame(band_err, index=band_labels)
print('\n=== summed |spectral error| per band (final time) ===')
print(band_df.to_string())


## 11. Notes

- **Four runs, one problem**: same sawtooth Burgers case as Chapter 1's TD-PINNs vs Vanilla
  notebook; the new axis of comparison is the **loss form**
  (`weak_projection` vs `agnostic`), crossed with the training **scheme** (TD-PINNs vs Vanilla).
  All four share collocation strategy, optimiser schedule, and architecture width — only the
  loss-form aggregation differs within each scheme.
- **`weak_projection`** (`pinns_framework/loss.py`) reuses the *same* strong pointwise
  `residual_fn` as a strong-form run — only the reduction from residual to scalar loss changes
  (Petrov–Galerkin projection onto `N_TEST` Legendre test functions instead of pointwise MSE).
- **`agnostic`** hands full control of the scalar loss to `residual_fn` — `PINLossFunction`
  performs no reduction at all. Both `agnostic` losses here implement the **full $\nu>0$
  reduced-order weak form**: integrating the diffusion term $\nu u_{xx}$ (and the convective flux
  $uu_x=F(u)_x$, $F(u)=u^2/2$) by parts against a Legendre test function that vanishes at the
  spatial boundary removes the boundary-flux term, so only first-order derivatives ($u_x$, and
  $u_t$ for Vanilla) are ever computed — the "reduced-order" property requested for this mode.
- **TD-PINNs vs Vanilla quadrature is asymmetric by construction, not by choice**: TD-PINNs
  collocation points already all share one time level (time is the outer stepping loop, not a
  network input), so `td_agnostic_loss` gets an exact sorted-trapezoid quadrature in $x$ for free.
  `PINTrainer._fit_vanilla` instead samples an independent random $t$ per collocation point
  (`t_col = torch.rand(n_col, 1) * T`), so no group of points shares a $t$ to integrate over — a
  per-slice spatial quadrature isn't available. `vanilla_agnostic_loss` works around this by
  testing against a product basis $v_k(x)\psi_m(t)$ and Monte-Carlo-integrating the whole
  space–time rectangle at once (same MC convention as `weak_projection`, extended to 2-D). See
  Section 3 for the full derivation of both.
- **Two known MC pitfalls in `vanilla_agnostic_loss`, both corrected**: (1) raw Legendre test
  functions have degree-dependent $L^2$ norms, so a product basis without normalisation would let
  low-order modes dominate the loss — fixed by scaling $v_k,\psi_m$ to unit norm (closed-form,
  applied in `_vanishing_tests`/`_legendre_plain`, benefits `td_agnostic_loss` too). (2) squaring a
  single-batch Monte Carlo estimate of $R_{km}$ is a biased estimate of $R_{km}^2$ — the bias is
  the estimator's own variance, which doesn't vanish at the true solution and can dominate once
  the residual itself gets small — fixed with the two-sample (U-statistic) trick: split the batch
  in half, multiply the two independent halves' estimates instead of squaring one. The resulting
  per-batch loss can occasionally dip slightly negative (expected, harmless for SGD); don't use it
  as a convergence metric — `compute_metrics` against the analytical solution, used throughout
  this notebook, is unaffected.
- **Errors**: $L_1$/$L_2$/$L_\infty$ vs the analytical sawtooth, per time level — saved to
  `figs/burgers_weakforms_errors.pdf`.
- **Conservation**: mass is analytically constant ($4\cdot2\pi\approx25.13$); energy decreases by
  viscous dissipation — `figs/burgers_weakforms_conservation.pdf`. Both `agnostic` losses now
  include the diffusion term, so their energy curves should dissipate correctly too.
- **Spectral test**: FFT-in-$x$ error split into DC / low / mid / high bands at the final time —
  `figs/burgers_weakforms_spectral.pdf`.
- Tune `TD_EPOCHS`, `VAN_EPOCHS`, `DT`, `NT`, `N_COLLOC`, `N_TEST`, `N_TEST_T` in the config cell
  (Section 2). `VAN_TOL_AGN` there is intentionally an unreachable tolerance, not a typo — see
  its comment.
- A cheap sanity check worth adding if you extend this notebook: for small `DT`, `td_agnostic_loss`
  and `vanilla_agnostic_loss` implement the *same* spatial weak form (only the time discretisation
  differs — discrete CN vs. autodiff), so their residuals should agree closely on the same
  reference solution.
